# Experiment 03 — MEMFOF zero-shot master baseline

Same shape as Experiment 01 (`notebooks/raft-sintel-master-baseline/`): discover the Spring
test frames, run native-resolution inference, validate the prediction tree, and package it
with the official subsampler. Only the model changes.

## Why this experiment

| Rank | Method | 1px | |
| ---: | --- | ---: | --- |
| 15 | **MEMFOF, no Spring training** | **3.600** | what this notebook runs |
| 45 | RoCo-45 — RAFT-Sintel (Experiment 01) | 6.766 | where we are |

MEMFOF's published zero-shot checkpoint scores roughly **half our current error** without
training on Spring at all. See `guides/modern-models-scoping.md` for the leaderboard snapshot
and the porting evidence behind this notebook.

`tskh` is the **Tartan-T-TSKH** checkpoint — TartanAir, Things, then a Sintel/KITTI/HD1K
mixture. It has never seen Spring, so this is a genuine zero-shot result. (A `spring`
checkpoint also exists; using it would not be zero-shot, and its leaderboard number is
already published.)

## Required Kaggle inputs

| Input | Used for |
| --- | --- |
| Spring **test** frames, left and right | the 1,980 image pairs to predict |
| `flow_subsampling` binary | packaging the `.hdf5` artifact |
| Internet **On** | PTLFlow source, the MEMFOF checkpoint, torchvision ResNet-34 |

## Three things that differ from the RAFT notebook

1. **MEMFOF is not in the pinned devkit.** Cell 4 runs `guides/port_ptlflow_models.py`, which
   copies the model out of upstream PTLFlow (pinned) and rewrites two import prefixes plus one
   renamed dict key. Ten files, one rewritten.
2. **MEMFOF reads three frames**, so the dataset token is `spring-seqlen_3`. The devkit pads
   sequence ends, so the sample count is unchanged and the output filenames still key off the
   first frame of each window.
3. **Two inference passes.** RAFT produced both directions in one pass via
   `--model.predict_all_directions`. MEMFOF has no such argument and returns only `flows` at
   inference — `test.py` writes a backward file only when `flows_b` is present. The second
   pass therefore uses `timerevonly`, which makes the dataset run time-reversed and makes
   `_generate_output_paths` label the result `BW`. Both passes write into the same tree.

There is no `--model.corr_mode` here. `triton` is a fork-local addition to the devkit's RAFT;
passing it to MEMFOF is an error.

## 1. Freeze the experiment configuration

In [ ]:
from pathlib import Path
import hashlib, json, os, platform, re, shutil, subprocess, sys, time

TEAM_ID, TEAM_NAME = "RoCo-45", "Flow State"
MODEL = "memfof"
CHECKPOINT_ALIAS = "tskh"        # Tartan-T-TSKH: no Spring training, so genuinely zero-shot
ITERATIONS = 8                   # MEMFOF's own default and its published inference setting
SEQUENCE_LENGTH = 3              # MEMFOF consumes a 3-frame window
NUM_GPUS = 2
MAX_FORWARD_SIDE = None          # native 1920x1080; None is mandatory here
DEVKIT_REF = "90ae81a9324c6806dc3c2482aab84a2744215bd9"
PTLFLOW_REF = "b4e897c6b0c11d695fa9a35de1a728354890420d"

KAGGLE_INPUT = Path("/kaggle/input")
WORK_ROOT = Path("/kaggle/working")
SCRATCH_ROOT = Path("/kaggle/temp")
DEVKIT_DIR = WORK_ROOT / "roco-spring-devkit"
OUTPUT_BASE = SCRATCH_ROOT / "exp03_predictions"
ARTIFACT_DIR = WORK_ROOT / "exp03_artifacts"

# Both passes; the second is what produces the backward flows.
PASSES = [
    ("FW", f"spring-seqlen_{SEQUENCE_LENGTH}"),
    ("BW", f"spring-seqlen_{SEQUENCE_LENGTH}-timerevonly"),
]
EXPECTED_FLOW_FILES = 3960       # 990 frames x {FW,BW} x {left,right}

SESSION_START = time.monotonic()
MAX_SESSION_HOURS = 12.0
PACKAGING_RESERVE_MINUTES = 45

assert KAGGLE_INPUT.is_dir(), "Run this notebook in Kaggle."
config = {
    "team": f"{TEAM_ID} - {TEAM_NAME}",
    "model": "MEMFOF (multi-frame, 3-frame window)",
    "checkpoint": CHECKPOINT_ALIAS,
    "spring_training": False,
    "iterations": ITERATIONS,
    "input_resolution": "native 1920x1080 (no rescaling)",
    "gpus": NUM_GPUS,
    "fine_tuning": False,
    "test_time_augmentation": False,
    "passes": [t for _, t in PASSES],
}
print(json.dumps(config, indent=2))

## 2. Hardware and storage preflight

Fail now rather than an hour in. MEMFOF's published 1080p inference memory is **2.09 GB**, so
a 15 GB T4 is not close to the limit — but the prediction tree is large, and it goes to
`/kaggle/temp`, not `/kaggle/working`.

In [ ]:
gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
                      "--format=csv,noheader"], text=True, capture_output=True, check=True)
gpus = [l.strip() for l in gpu.stdout.splitlines() if l.strip()]
print("\n".join(gpus))
if len(gpus) < 1:
    raise RuntimeError("Enable a GPU accelerator and restart the session.")

SCRATCH_ROOT.mkdir(parents=True, exist_ok=True)
for p in (WORK_ROOT, SCRATCH_ROOT):
    free = shutil.disk_usage(p).free / 1024**3
    print(f"{str(p):18s} {free:7.1f} GiB free")

# Experiment 01's tree of the same 3,960 files was about 46 GiB before packaging.
if shutil.disk_usage(SCRATCH_ROOT).free / 1024**3 < 60:
    raise RuntimeError("Less than 60 GiB on /kaggle/temp; the prediction tree will not fit.")

record = {"python": sys.version.split()[0], "platform": platform.platform(), "gpus": gpus}
print(json.dumps(record, indent=2))

## 3. Discover and verify the Spring test frames

Located by searching for the directory layout rather than a hard-coded dataset slug, because
Kaggle does not guarantee the published path. `test.py` needs a single `spring_root_dir` whose
`test/` holds every sequence, so separately attached left/right/cam datasets are symlinked
into one tree — links only, no copying.

In [ ]:
import glob

parts = sorted({Path(p).parents[2] for p in
                glob.glob(str(KAGGLE_INPUT / "**/test/*/frame_left"), recursive=True)}
               | {Path(p).parents[2] for p in
                  glob.glob(str(KAGGLE_INPUT / "**/test/*/frame_right"), recursive=True)})
if not parts:
    print("No Spring test tree under /kaggle/input. What IS attached:")
    for p in sorted(glob.glob(str(KAGGLE_INPUT / "*"))):
        print("   ", p)
    raise RuntimeError("Attach the Spring test frames (left and right).")

SPRING_ROOT = WORK_ROOT / "spring_test_root"
(SPRING_ROOT / "test").mkdir(parents=True, exist_ok=True)
for part in parts:
    for seq in sorted((part / "test").iterdir()):
        if not seq.is_dir():
            continue
        link = SPRING_ROOT / "test" / seq.name
        link.mkdir(exist_ok=True)
        for sub in sorted(seq.iterdir()):
            target = link / sub.name
            if not target.exists():
                target.symlink_to(sub)

sequences = sorted((SPRING_ROOT / "test").iterdir())
n_left = sum(len(list((s / "frame_left").glob("*.png"))) for s in sequences
             if (s / "frame_left").is_dir())
n_right = sum(len(list((s / "frame_right").glob("*.png"))) for s in sequences
              if (s / "frame_right").is_dir())
print(f"assembled {len(sequences)} test sequences at {SPRING_ROOT}")
print(f"  frame_left  {n_left:5d}")
print(f"  frame_right {n_right:5d}")
assert n_left == n_right and n_left > 0, "left and right frame counts must match"

# One pair per adjacent frame, per side: (n_left - n_sequences) x 2 sides.
expected_pairs = (n_left - len(sequences)) * 2
print(f"  expected pairs per pass: {expected_pairs}")
assert expected_pairs * 2 == EXPECTED_FLOW_FILES, (
    f"two passes over {expected_pairs} pairs give {expected_pairs*2} files, "
    f"not the {EXPECTED_FLOW_FILES} the benchmark requires")

## 4. Install the devkit, then port MEMFOF into it

The devkit is a trimmed fork of PTLFlow that ships only RAFT, but its `BaseModel` interface is
unchanged from upstream, so a model is copied rather than reimplemented. The script applies
three substitutions:

```
ptlflow.utils.registry  ->  roco_spring_devkit.common.utils.registry
ptlflow.utils.utils     ->  roco_spring_devkit.common.utils.utils
inputs["valids"]        ->  inputs["valid_flows"]
```

The third matters only for training — inference works without it — so it would not surface
here, but the port is kept faithful so the same devkit can be fine-tuned later.

This does not survive a re-clone. Run it once per session, before importing any model.

In [ ]:
%%bash
set -euo pipefail
cd /kaggle/working
rm -rf roco-spring-devkit
git clone -q https://github.com/hmorimitsu/roco-spring-devkit.git
cd roco-spring-devkit
git checkout -q 90ae81a9324c6806dc3c2482aab84a2744215bd9
pip install -q -e .

In [ ]:
# port_ptlflow_models.py lives in this repo. Attach the repo as a dataset, or paste the
# script into the session; the fallback below fetches it from the working copy if present.
PORT_SCRIPT = None
for cand in list(KAGGLE_INPUT.rglob("port_ptlflow_models.py")) + [WORK_ROOT / "port_ptlflow_models.py"]:
    if Path(cand).is_file():
        PORT_SCRIPT = Path(cand)
        break
if PORT_SCRIPT is None:
    raise RuntimeError(
        "port_ptlflow_models.py not found. Attach the FlowState repo as a Kaggle dataset, "
        "or upload guides/port_ptlflow_models.py into /kaggle/working.")

subprocess.run([sys.executable, str(PORT_SCRIPT), "--devkit", str(DEVKIT_DIR),
                "--models", "memfof"], check=True)

devkit_path = str(DEVKIT_DIR.resolve())
if devkit_path not in sys.path:
    sys.path.insert(0, devkit_path)

from roco_spring_devkit.optical_flow.models.memfof.memfof import MEMFOF
probe = MEMFOF(iters=ITERATIONS)
parameter_count = sum(p.numel() for p in probe.parameters())
print(f"MEMFOF ready: {parameter_count/1e6:.2f} M parameters, output_stride {probe.output_stride}")
assert MEMFOF.pretrained_checkpoints[CHECKPOINT_ALIAS], "checkpoint alias not registered"
expected_checkpoint_url = MEMFOF.pretrained_checkpoints[CHECKPOINT_ALIAS]
print("checkpoint:", expected_checkpoint_url)
del probe

## 5. Optional — rank it on sequence 0022 before spending a submission

Leaderboard submissions are budgeted in `new_plan.md`, and Experiment 01 established that our
local `0022` number tracks the test set closely. If the training dataset happens to be
attached, this measures MEMFOF on the same 18 pairs our RAFT runs used, for about a minute of
GPU. It is skipped silently when the data is absent.

Bear in mind that `0022` is 18 pairs and its standard error is roughly 4.6 — it separates
*architectures*, not small tweaks.

In [ ]:
val_hits = sorted(KAGGLE_INPUT.glob("**/train/0022/flow_FW_left"))
if not val_hits:
    print("training data not attached - skipping the local check")
else:
    TRAIN_ROOT = str(val_hits[0]).replace("/train/0022/flow_FW_left", "")
    print("validating against", TRAIN_ROOT)
    subprocess.run(
        [sys.executable, "-u", "validate.py",
         "--data.val_dataset", f"spring-val-left-seqlen_{SEQUENCE_LENGTH}",
         "--data.spring_root_dir", TRAIN_ROOT,
         "--model", MODEL,
         "--ckpt_path", CHECKPOINT_ALIAS,
         "--model.iters", str(ITERATIONS)],
        cwd=DEVKIT_DIR / "roco_spring_devkit" / "optical_flow", check=False)

## 6. Native-resolution inference, both directions

Two passes over the same output path. The first writes `flow_FW_{left,right}`; the second runs
the dataset time-reversed, which `_generate_output_paths` labels `BW`.

`--max_forward_side` and `--scale_factor` are deliberately absent: the benchmark requires
predictions at 1920x1080, and `BaseModel` already pads to MEMFOF's stride of 32 and unpads
afterwards.

In [ ]:
OPTICAL_FLOW_DIR = DEVKIT_DIR / "roco_spring_devkit" / "optical_flow"
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

env = os.environ.copy()
env["CUDA_VISIBLE_DEVICES"] = ",".join(str(i) for i in range(NUM_GPUS))
env["PYTHONUNBUFFERED"] = "1"
env["PYTHONPATH"] = devkit_path + (os.pathsep + env["PYTHONPATH"] if env.get("PYTHONPATH") else "")

for direction, token in PASSES:
    cmd = [
        sys.executable, "-u", "test.py",
        "--data.test_dataset", token,
        "--data.spring_root_dir", str(SPRING_ROOT),
        "--model", MODEL,
        "--ckpt_path", CHECKPOINT_ALIAS,
        "--model.iters", str(ITERATIONS),
        "--num_gpus", str(NUM_GPUS),
        "--output_path", str(OUTPUT_BASE),
    ]
    assert MAX_FORWARD_SIDE is None
    assert not any(a in cmd for a in ("--max_forward_side", "--scale_factor", "--model.corr_mode"))
    print(f"\n===== {direction} pass =====")
    print(" ".join(cmd))

    remaining = MAX_SESSION_HOURS * 3600 - PACKAGING_RESERVE_MINUTES * 60 \
        - (time.monotonic() - SESSION_START)
    if remaining <= 0:
        raise TimeoutError("No safe inference time remains. Restart the session.")
    subprocess.run(cmd, cwd=OPTICAL_FLOW_DIR, env=env, check=True, timeout=remaining)

    n = len(list(OUTPUT_BASE.rglob(f"flow_{direction}_*.flo5")))
    print(f"{direction} pass wrote {n} files")

## 7. Validate the prediction tree before packaging

The same four checks Experiment 01 used — filename pattern, file count, native shape, finite
values — plus one this notebook needs: that both directions and both camera sides are present.
A submission that is complete but silently half-backward would be indistinguishable from a
good one until the leaderboard rejected it.

In [ ]:
import h5py
import numpy as np

roots = [p for p in OUTPUT_BASE.glob("*/spring") if any(p.rglob("*.flo5"))]
if len(roots) != 1:
    raise RuntimeError(f"Expected exactly one Spring prediction tree; found {roots}")
PREDICTION_ROOT = roots[0]

files = sorted(PREDICTION_ROOT.rglob("*.flo5"))
if len(files) != EXPECTED_FLOW_FILES:
    raise RuntimeError(f"Expected {EXPECTED_FLOW_FILES} flow files; found {len(files)}")

name_re = re.compile(r"flow_(FW|BW)_(left|right)_\d{4}\.flo5$")
directions = {}
for path in files:
    m = name_re.fullmatch(path.name)
    if not m:
        raise RuntimeError(f"Invalid prediction filename: {path}")
    directions[m.groups()] = directions.get(m.groups(), 0) + 1

required = {(d, s) for d in ("FW", "BW") for s in ("left", "right")}
if set(directions) != required:
    raise RuntimeError(f"Expected FW/BW x left/right; found {sorted(directions)}")
for key in sorted(directions):
    print(f"  {key[0]}_{key[1]:5s} {directions[key]:5d} files")
if len(set(directions.values())) != 1:
    raise RuntimeError(f"Direction/side counts are not equal: {directions}")

for index in sorted({0, len(files) // 3, 2 * len(files) // 3, len(files) - 1}):
    with h5py.File(files[index], "r") as h:
        if "flow" not in h or h["flow"].shape != (1080, 1920, 2):
            raise RuntimeError(f"Bad shape in {files[index]}: "
                               f"{h['flow'].shape if 'flow' in h else None}")
        if not np.isfinite(h["flow"][:]).all():
            raise RuntimeError(f"NaN or Inf in {files[index]}")

size_gib = sum(p.stat().st_size for p in files) / 1024**3
print(f"\nvalidated {len(files):,} native-resolution .flo5 files ({size_gib:.2f} GiB)")
print("prediction root:", PREDICTION_ROOT)

## 8. Package the benchmark artifact and write the manifest

`flow_subsampling` writes into the current directory, so it runs with `cwd` set to the
artifact directory. The manifest follows `experiment_01_manifest.json` so the two experiments
can be compared field by field.

In [ ]:
tools = sorted(KAGGLE_INPUT.rglob("flow_subsampling"))
if not tools:
    raise RuntimeError("flow_subsampling not found under /kaggle/input - attach it.")
TOOL = tools[0]
TOOL.chmod(0o755)

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([str(TOOL), str(PREDICTION_ROOT)], cwd=ARTIFACT_DIR, check=True)

artifacts = sorted(ARTIFACT_DIR.glob("*.hdf5"), key=lambda p: p.stat().st_mtime)
if not artifacts:
    raise RuntimeError("flow_subsampling produced no HDF5 artifact.")
SUBMISSION_FILE = artifacts[-1]

def sha256(path):
    d = hashlib.sha256()
    with path.open("rb") as fh:
        for chunk in iter(lambda: fh.read(8 * 1024 * 1024), b""):
            d.update(chunk)
    return d.hexdigest()

manifest = {
    "team_id": TEAM_ID,
    "team_name": TEAM_NAME,
    "experiment": "03_memfof_tskh_zeroshot_native",
    "model": MODEL,
    "model_class": "MEMFOF (multi-frame, 3-frame window)",
    "parameter_count": parameter_count,
    "checkpoint": CHECKPOINT_ALIAS,
    "checkpoint_url": expected_checkpoint_url,
    "spring_training": False,
    "fine_tuning": False,
    "test_time_augmentation": False,
    "iterations": ITERATIONS,
    "sequence_length": SEQUENCE_LENGTH,
    "dataset_tokens": [t for _, t in PASSES],
    "resolution": "1920x1080_native",
    "devkit_ref": DEVKIT_REF,
    "ptlflow_ref": PTLFLOW_REF,
    "prediction_files": len(files),
    "submission_file": SUBMISSION_FILE.name,
    "submission_bytes": SUBMISSION_FILE.stat().st_size,
    "submission_sha256": sha256(SUBMISSION_FILE),
}
path = ARTIFACT_DIR / "experiment_03_manifest.json"
path.write_text(json.dumps(manifest, indent=2) + "\n")
print(path.read_text())
print("UPLOAD THIS FILE:", SUBMISSION_FILE)

## Submission checklist

1. `Save Version -> Save & Run All`, then download the `.hdf5` and `experiment_03_manifest.json`.
2. Commit the manifest to `notebooks/exp03-memfof-zeroshot/` and add a row to `STATUS.md`.
3. Upload the `.hdf5` to the Spring optical-flow benchmark.

Declare honestly on the submission form: MEMFOF's **Tartan-T-TSKH** checkpoint, no Spring
training, no test-time augmentation, native resolution. The benchmark requires disclosing
pretraining data and checkpoints, and this run uses a public checkpoint trained on public data.

If the artifact is uploaded, record the returned score next to the local `0022` number from
cell 5 — that pair is what tells us how far local validation can be trusted for the *next*
architecture, which is worth more than this single submission.